Monday: the warehouse answers, and it answers differently

> "I want these numbers every Monday, for every segment and channel, computed from the warehouse
> itself. No notebooks, no exports, nothing a person can mistype."
>
> Anand Iyer, finance controller, replying to your growth note.

Last week you answered Meera from a file. This notebook is the bridge: the same tree, asked of a
database, and the one number that comes back different.

**MAP** what a query is  ->  **DO** filter and group  ->  **SEE** the error  ->
**CHECK** the suite  ->  **SUM** what changed.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
print("connected to", kit.WAREHOUSE["dbname"], "on", kit.WAREHOUSE["host"])

connected to kalpa on 127.0.0.1


## MAP. A query describes the result, and the server decides how

Last week's accumulator said *how*: open the file, walk the rows, add to a total. A query says
*what*: one row per segment, carrying a sum. The server picks the walk.

That is why the SQL is shorter than the Python it replaces, and why nothing in it tells you the
order rows come back in.

In [2]:
kit.flow(["you write SQL", "the planner", "the rows", "your result"],
         lit=[1], title="Where the decisions are made")

## DO. What is in here, before anything is asked of it

A column you have not looked at is a column you will misuse. Six tables, and today needs two.

In [3]:
kit.sql_table("""
    SELECT table_name, count(*) AS columns
    FROM information_schema.columns
    WHERE table_schema = 'public'
    GROUP BY table_name ORDER BY table_name
""", caption="The warehouse, table by table", conn=conn)

table_name,columns
campaign_exposure,3
campaigns,5
customers,5
orders,7
payments,6
plan_line,2
refunds,5


## DO. The book is bigger than the file

The platform lead said one thousand orders. Last week's extract held two hundred rows. Run the
headline both ways and hold the two numbers side by side.

In [4]:
rows = kit.sql("SELECT quarter, count(*) AS orders, sum(amount) AS revenue "
               "FROM orders GROUP BY quarter ORDER BY quarter", conn=conn)
for r in rows:
    print(f"{r['quarter']}  {r['orders']:>4} orders  {kit.rupees(r['revenue'])}")

last_week = {"Q1": 19000000, "Q2": 18700000}
print()
for r in rows:
    ratio = float(r["revenue"]) / last_week[r["quarter"]]
    print(f"{r['quarter']}: the warehouse is {ratio:.2f} times last week's note")

Q1   538 orders  Rs 10,00,00,000
Q2   462 orders  Rs 9,84,00,000

Q1: the warehouse is 5.26 times last week's note
Q2: the warehouse is 5.26 times last week's note


### The reconciliation, in one sentence

Neither number is wrong. Last week's file was an extract, so its levels were a sample's levels.
What the sample got right was the *shape*: which segment moved, and in which direction.

Check the shape survives before trusting anything else.

In [5]:
shape = kit.sql("""
    WITH q1 AS (SELECT c.segment, count(*) n FROM orders o
                JOIN customers c USING (customer_id)
                WHERE o.quarter='Q1' GROUP BY c.segment),
         q2 AS (SELECT c.segment, count(*) n FROM orders o
                JOIN customers c USING (customer_id)
                WHERE o.quarter='Q2' GROUP BY c.segment)
    SELECT q1.segment, q1.n AS q1, q2.n AS q2,
           round(100.0*(q2.n-q1.n)/q1.n, 1) AS pct
    FROM q1 JOIN q2 USING (segment) ORDER BY pct
""", conn=conn)
kit.table(["segment", "Q1 orders", "Q2 orders", "change %"],
          [[r["segment"], r["q1"], r["q2"], f'{r["pct"]}%'] for r in shape],
          caption="Retail-Plus is still the branch that moved")

segment,Q1 orders,Q2 orders,change %
Retail-Plus,215,140,-34.9%
Business,97,91,-6.2%
Retail-Core,199,193,-3.0%
Student,27,38,40.7%


In [6]:
plus = next(r for r in shape if r["segment"] == "Retail-Plus")
core = next(r for r in shape if r["segment"] == "Retail-Core")
kit.check("Retail-Plus order count falls by more than a quarter", float(plus["pct"]) < -25,
          f'{plus["pct"]}%')
kit.check("Retail-Core barely moves", abs(float(core["pct"])) < 6, f'{core["pct"]}%')
kit.check("the sample's finding survives at book scale", float(plus["pct"]) < float(core["pct"]))

## SEE. The error worth meeting on purpose

Anand wants the numbers per segment **and** per channel. Ask for both while grouping by one and
the server refuses, because it has one row per segment to fill and many channels to choose from.

In [7]:
try:
    kit.sql("""SELECT c.segment, o.channel, sum(o.amount)
               FROM orders o JOIN customers c USING (customer_id)
               GROUP BY c.segment""", conn=conn)
except Exception as e:
    conn.rollback()
    print(type(e).__name__)
    print(str(e).strip())

GroupingError
column "o.channel" must appear in the GROUP BY clause or be used in an aggregate function
LINE 1: SELECT c.segment, o.channel, sum(o.amount)
                          ^


### Two fixes, and they answer different questions

The message says exactly what to do, and it leaves the choice to you. Adding the column to the
grouping gives one row per pair. Wrapping it in an aggregate keeps four rows and answers a
different question.

In [8]:
wide = kit.sql("""SELECT c.segment, o.channel, count(*) n
                  FROM orders o JOIN customers c USING (customer_id)
                  GROUP BY c.segment, o.channel""", conn=conn)
narrow = kit.sql("""SELECT c.segment, count(DISTINCT o.channel) channels
                    FROM orders o JOIN customers c USING (customer_id)
                    GROUP BY c.segment""", conn=conn)
print(f"adding channel to GROUP BY   -> {len(wide)} rows")
print(f"wrapping it in an aggregate  -> {len(narrow)} rows")
kit.check("grouping wider returns more rows than grouping narrow", len(wide) > len(narrow),
          f"{len(wide)} against {len(narrow)}")

adding channel to GROUP BY   -> 12 rows
wrapping it in an aggregate  -> 4 rows


## SEE. The order a query runs in is not the order you write it

Almost every error today is this picture. `WHERE` runs before any group exists, which is why an
aggregate is refused there. `SELECT` runs after grouping, which is why an alias made there is
invisible to `WHERE` and visible to `ORDER BY`.

In [9]:
kit.vflow(["FROM: get the rows", "WHERE: drop rows", "GROUP BY: form groups",
           "HAVING: drop groups", "SELECT: compute columns", "ORDER BY: sort", "LIMIT: cut"],
          lit=[1, 3], title="The logical execution order")

In [10]:
try:
    kit.sql("SELECT quarter, count(*) FROM orders WHERE count(*) > 100 GROUP BY quarter",
            conn=conn)
except Exception as e:
    conn.rollback()
    print(str(e).strip().splitlines()[0])
    print("\nThe fix is HAVING, which runs after the groups exist.")

having = kit.sql("SELECT quarter, count(*) n FROM orders GROUP BY quarter HAVING count(*) > 100",
                 conn=conn)
kit.check("HAVING accepts the same condition WHERE refused", len(having) == 2, f"{len(having)} rows")

aggregate functions are not allowed in WHERE

The fix is HAVING, which runs after the groups exist.


## CHECK. Two named steps, and the deliverable

A three-level nested subquery is correct and unreadable, and Anand's analyst audits every line.
A CTE is a named step: one block, one job, named after the job.

In [11]:
kit.sql_table("""
    WITH q1 AS (SELECT c.segment, count(*) orders, sum(o.amount) revenue
                FROM orders o JOIN customers c USING (customer_id)
                WHERE o.quarter='Q1' GROUP BY c.segment),
         q2 AS (SELECT c.segment, count(*) orders, sum(o.amount) revenue
                FROM orders o JOIN customers c USING (customer_id)
                WHERE o.quarter='Q2' GROUP BY c.segment)
    SELECT q1.segment, q1.orders AS q1_orders, q2.orders AS q2_orders,
           round(100.0*(q2.orders-q1.orders)/q1.orders, 1) AS order_change_pct
    FROM q1 JOIN q2 USING (segment) ORDER BY order_change_pct
""", caption="Query six of the Monday suite", conn=conn)

segment,q1_orders,q2_orders,order_change_pct
Retail-Plus,215,140,-34.9
Business,97,91,-6.2
Retail-Core,199,193,-3
Student,27,38,40.7


In [12]:
total = kit.sql("SELECT sum(amount) s FROM orders", conn=conn)[0]["s"]
parts = kit.sql("""SELECT sum(o.amount) s FROM orders o
                   JOIN customers c USING (customer_id)""", conn=conn)[0]["s"]
kit.check("every order belongs to a customer that exists", total == parts,
          f"{kit.rupees(total)} both ways")
kit.check("the two quarters add to the whole book",
          sum(r["revenue"] for r in rows) == total, kit.rupees(total))

## SUM. What changed today

The analysis did not change. Where it lives did.

In [13]:
kit.matrix(["last week", "this Monday"],
           ["source", "who can run it", "when it is right"],
           [["a 200-row file", "you", "the day it was pulled"],
            ["the warehouse", "anybody with read access", "every time it runs"]],
           title="What Anand actually bought")
kit.check_summary()